# LoRA: сколько Transformer-блоков адаптировать
**Вопрос:** нужно ли ставить LoRA во все блоки BERT, или хватает последних 25/50/75%?

Ноутбук самодостаточный: весь код внутри, ничего клонировать не нужно. Выполняй ячейки сверху вниз:
среда → код → данные → модель и конфиг LoRA → пробный запуск → вся сетка → метрики → графики → скачать результаты.

Перед запуском: *Runtime → Change runtime type → T4 GPU*.

## Шаг 1. Среда
На Colab torch уже установлен. Ставим peft (LoRA), обновляем transformers. Версии и GPU печатаем: их нужно вписать в раздел «Воспроизводимость» отчёта.

In [ ]:
!pip -q install -U transformers peft
import torch, transformers, peft, platform
print("python", platform.python_version(), "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| peft", peft.__version__)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (включи GPU!)")

## Шаг 2. Код эксперимента
`lora_depth.py` — один запуск: загрузка SST-2 → модель → LoRA в выбранных блоках → обучение → метрики → строка в `results/results.csv`.

Главное место — функция `build_model`: `LoraConfig(layers_to_transform=layers, ...)` ставит LoRA только в выбранные блоки.

In [ ]:
%%writefile lora_depth.py
"""
Сколько Transformer-блоков адаптировать LoRA?

Один запуск = одна конфигурация (доля блоков, позиция, seed).
LoRA ставится только в выбранные блоки энкодера через PEFT `layers_to_transform`.
Классификационная голова (pooler + classifier) обучается во всех вариантах одинаково.

Варианты (--mode):
  head  : backbone заморожен полностью, учится только голова (0% блоков)  -> нижний baseline
  lora  : LoRA в доле --fraction блоков (последних или первых, --position)
  full  : полный fine-tuning всех весов                                      -> верхний ориентир

Пример:
  python lora_depth.py --mode lora --fraction 0.5 --seed 0
"""
import argparse
import csv
import json
import math
import os
import random
import time
import urllib.request

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

SST2_URL = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/{}.tsv"


# ----------------------------------------------------------------------------- data
def load_sst2(data_dir):
    """SST-2 (sentence level, binary): train 6920 / dev 872 / test 1821, с метками у test."""
    os.makedirs(data_dir, exist_ok=True)
    splits = {}
    for split in ("train", "dev", "test"):
        path = os.path.join(data_dir, f"{split}.tsv")
        if not os.path.exists(path):
            urllib.request.urlretrieve(SST2_URL.format(split), path)
        texts, labels = [], []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                text, label = line.rsplit("\t", 1)
                texts.append(text)
                labels.append(int(label))
        splits[split] = (texts, labels)
    return splits


def make_loader(tok, texts, labels, max_len, batch_size, shuffle, seed):
    enc = tok(texts, truncation=True, max_length=max_len)
    items = [
        {"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": labels[i]}
        for i in range(len(texts))
    ]

    def collate(batch):
        L = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for j, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[j, :n] = torch.tensor(b["input_ids"])
            att[j, :n] = 1
        return {"input_ids": ids, "attention_mask": att,
                "labels": torch.tensor([b["labels"] for b in batch])}

    g = torch.Generator().manual_seed(seed)
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate, generator=g)


# ----------------------------------------------------------------------------- model
def select_layers(n_layers, fraction, position):
    k = max(1, round(n_layers * fraction))
    return list(range(n_layers - k, n_layers)) if position == "last" else list(range(k))


def build_model(args):
    model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2)
    # размер исходной модели (backbone + pooler + classifier) до добавления LoRA;
    # после get_peft_model total вырастет за счёт LoRA-матриц и копий pooler/classifier
    # из modules_to_save, поэтому долю обучаемых параметров считаем от этого числа
    model.base_param_count = sum(p.numel() for p in model.parameters())
    n_layers = model.config.num_hidden_layers
    layers = []
    if args.mode == "full":
        pass  # все веса обучаются
    elif args.mode == "head":
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("classifier") or ".pooler." in name or name.startswith("bert.pooler")
    elif args.mode == "lora":
        layers = select_layers(n_layers, args.fraction, args.position)
        cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=args.rank,
            lora_alpha=args.alpha,
            lora_dropout=0.1,
            # все линейные слои блока: Q, K, V, выход attention, FFN up/down
            target_modules=["query", "key", "value", "dense"],
            layers_to_transform=layers,
            layers_pattern="layer",
            modules_to_save=["classifier", "pooler"],
        )
        base = model.base_param_count
        model = get_peft_model(model, cfg)
        model.base_param_count = base
    return model, n_layers, layers


def count_params(model):
    """trainable — обучаемые параметры; total — все параметры объекта (с LoRA и копиями modules_to_save)."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = n = 0
    loss_sum = 0.0
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out = model(**b)
        loss_sum += out.loss.item() * len(b["labels"])
        correct += (out.logits.argmax(-1) == b["labels"]).sum().item()
        n += len(b["labels"])
    return correct / n, loss_sum / n


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="google/bert_uncased_L-8_H-512_A-8")
    ap.add_argument("--mode", choices=["head", "lora", "full"], default="lora")
    ap.add_argument("--fraction", type=float, default=1.0)
    ap.add_argument("--position", choices=["last", "first"], default="last")
    ap.add_argument("--rank", type=int, default=8)
    ap.add_argument("--alpha", type=int, default=16)
    ap.add_argument("--lr", type=float, default=None, help="по умолчанию: 5e-4 для lora/head, 5e-5 для full")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_len", type=int, default=64)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--data_dir", default="data/sst2")
    ap.add_argument("--out", default="results/results.csv")
    ap.add_argument("--threads", type=int, default=0)
    args = ap.parse_args()
    if args.lr is None:
        args.lr = 5e-5 if args.mode == "full" else 5e-4
    if args.threads:
        torch.set_num_threads(args.threads)

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)

    data = load_sst2(args.data_dir)
    tok = AutoTokenizer.from_pretrained(args.model)
    train_dl = make_loader(tok, *data["train"], args.max_len, args.batch_size, True, args.seed)
    dev_dl = make_loader(tok, *data["dev"], args.max_len, 128, False, args.seed)
    test_dl = make_loader(tok, *data["test"], args.max_len, 128, False, args.seed)

    model, n_layers, layers = build_model(args)
    trainable, total = count_params(model)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=0.01)
    steps = args.epochs * len(train_dl)
    warm = int(0.06 * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (steps - s) / max(1, steps - warm)))

    best = {"dev_acc": -1}
    history = []
    t0 = time.time()
    for ep in range(1, args.epochs + 1):
        model.train()
        tr_loss = 0.0
        for b in train_dl:
            b = {k: v.to(device) for k, v in b.items()}
            out = model(**b)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            tr_loss += out.loss.item()
        dev_acc, dev_loss = evaluate(model, dev_dl, device)
        history.append({"epoch": ep, "train_loss": tr_loss / len(train_dl), "dev_acc": dev_acc, "dev_loss": dev_loss})
        print(f"[ep {ep}] train_loss={tr_loss/len(train_dl):.4f} dev_acc={dev_acc:.4f} dev_loss={dev_loss:.4f}", flush=True)
        if dev_acc > best["dev_acc"]:
            # модель выбирается по dev, test считается только для лучшей эпохи
            test_acc, test_loss = evaluate(model, test_dl, device)
            best = {"dev_acc": dev_acc, "dev_loss": dev_loss, "test_acc": test_acc, "test_loss": test_loss, "best_epoch": ep}
    train_time = time.time() - t0

    frac = 0.0 if args.mode == "head" else (1.0 if args.mode == "full" else len(layers) / n_layers)
    row = {
        "model": os.path.basename(args.model.rstrip("/")), "mode": args.mode, "position": args.position if args.mode == "lora" else "",
        "fraction": round(frac, 4), "n_layers_adapted": len(layers) if args.mode == "lora" else (n_layers if args.mode == "full" else 0),
        "layers": " ".join(map(str, layers)), "rank": args.rank if args.mode == "lora" else "", "lr": args.lr,
        "seed": args.seed, "trainable_params": trainable, "total_params": total,
        "base_params": model.base_param_count,
        "trainable_pct": round(100 * trainable / model.base_param_count, 4), "train_time_s": round(train_time, 1), "device": str(device),
        **{k: (round(v, 5) if isinstance(v, float) else v) for k, v in best.items()},
        "history": json.dumps(history),
    }
    print(json.dumps({k: v for k, v in row.items() if k != "history"}, ensure_ascii=False))
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    new = not os.path.exists(args.out)
    with open(args.out, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new:
            w.writeheader()
        w.writerow(row)


if __name__ == "__main__":
    main()


`run_all.sh` — вся сетка: 7 вариантов × 3 seed = 21 запуск.

In [ ]:
%%writefile run_all.sh
#!/bin/bash
# Полная сетка эксперимента: 3 seed x (head, LoRA last 25/50/75/100%, full FT, LoRA first 50%)
MODEL=${MODEL:-google/bert_uncased_L-8_H-512_A-8}
OUT=${OUT:-results/results.csv}
for SEED in 0 1 2; do
  python lora_depth.py --model $MODEL --mode head --seed $SEED --out $OUT
  for F in 0.25 0.5 0.75 1.0; do
    python lora_depth.py --model $MODEL --mode lora --fraction $F --position last --seed $SEED --out $OUT
  done
  python lora_depth.py --model $MODEL --mode full --seed $SEED --out $OUT
  python lora_depth.py --model $MODEL --mode lora --fraction 0.5 --position first --seed $SEED --out $OUT
done


`analyze.py` — среднее ± std по seed, графики и готовые к вставке в LaTeX файлы `results_table.tex` и `numbers.tex`.

In [ ]:
%%writefile analyze.py
"""Агрегирует results.csv по seed и строит графики для отчёта.

python analyze.py --csv results/results.csv --out figures
"""
import argparse
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

LABELS = {
    ("head", "", 0.0): "Head only (0%)",
    ("lora", "last", 0.25): "LoRA last 25%",
    ("lora", "last", 0.5): "LoRA last 50%",
    ("lora", "last", 0.75): "LoRA last 75%",
    ("lora", "last", 1.0): "LoRA 100%",
    ("lora", "first", 0.5): "LoRA first 50%",
    ("full", "", 1.0): "Full fine-tuning",
}
ORDER = list(LABELS.values())


MACRO = {  # имя макроса в отчёте -> вариант
    "Head": "Head only (0%)", "LastTwentyFive": "LoRA last 25%", "LastFifty": "LoRA last 50%",
    "LastSeventyFive": "LoRA last 75%", "LoraFull": "LoRA 100%", "FirstFifty": "LoRA first 50%",
    "FullFT": "Full fine-tuning",
}


def fmt(x, d=1):
    return f"{x:.{d}f}".replace(".", "{,}")


def write_latex(g, out):
    """results_table.tex — готовая таблица; numbers.tex — макросы \\accX, \\stdX, \\paramsX, \\timeX для текста."""
    rows = []
    for v, r in g.iterrows():
        params = f"{int(r.params):,}".replace(",", "\\,")
        rows.append(
            f"{v.replace('%', chr(92) + '%')} & {fmt(r.test_mean*100)} $\\pm$ {fmt(r.test_std*100)} & "
            f"{fmt(r.dev_mean*100)} $\\pm$ {fmt(r.dev_std*100)} & {params} & {fmt(r.pct, 2)} & {r.time_mean:.0f} \\\\")
    n = int(g["n"].min())
    with open(os.path.join(out, "results_table.tex"), "w") as f:
        f.write("\\begin{table}[H]\n\\centering\n\\small\n"
                f"\\caption{{Основные результаты (среднее $\\pm$ std по {n} seed; accuracy в \\%)}}\n"
                "\\label{tab:results}\n\\begin{tabular}{lccrcc}\n\\toprule\n"
                "\\textbf{Вариант} & \\textbf{Test acc} & \\textbf{Dev acc} & \\textbf{Trainable} & "
                "\\textbf{\\% от модели} & \\textbf{Время, с} \\\\\n\\midrule\n"
                + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    with open(os.path.join(out, "numbers.tex"), "w") as f:
        f.write(f"\\newcommand{{\\nseeds}}{{{n}}}\n")
        for name, v in MACRO.items():
            if v not in g.index:
                continue
            r = g.loc[v]
            f.write(f"\\newcommand{{\\acc{name}}}{{{fmt(r.test_mean*100)}}}\n")
            f.write(f"\\newcommand{{\\std{name}}}{{{fmt(r.test_std*100)}}}\n")
            f.write(f"\\newcommand{{\\params{name}}}{{{int(r.params):,}}}\n".replace(",", "\\,"))
            f.write(f"\\newcommand{{\\time{name}}}{{{r.time_mean:.0f}}}\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="results/results.csv")
    ap.add_argument("--out", default="figures")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    df = pd.read_csv(a.csv, keep_default_na=False)
    df["fraction"] = df["fraction"].astype(float)
    # доля обучаемых — от исходной модели (без LoRA и копий modules_to_save);
    # для старых CSV без base_params исходная модель = total_params варианта head only / full
    if "base_params" not in df.columns or (df["base_params"] == "").any():
        df["base_params"] = df.loc[df["mode"].isin(["head", "full"]), "total_params"].astype(int).min()
    df["trainable_pct"] = 100 * df["trainable_params"].astype(int) / df["base_params"].astype(int)
    df["variant"] = [LABELS[(m, p, f)] for m, p, f in zip(df["mode"], df["position"], df["fraction"])]

    g = df.groupby("variant").agg(
        n=("seed", "count"),
        test_mean=("test_acc", "mean"), test_std=("test_acc", "std"),
        dev_mean=("dev_acc", "mean"), dev_std=("dev_acc", "std"),
        params=("trainable_params", "first"), pct=("trainable_pct", "first"),
        time_mean=("train_time_s", "mean"),
    ).reindex([v for v in ORDER if v in set(df["variant"])])
    g.to_csv(os.path.join(a.out, "summary.csv"))
    pd.set_option("display.width", 200)
    print(g.round(4))
    print("\nper-seed test acc:")
    print(df.pivot_table(index="variant", columns="seed", values="test_acc").reindex(g.index).round(4))
    write_latex(g, a.out)

    # --- рисунок 1: доля блоков -> качество
    curve = df[(df["mode"] == "head") | ((df["mode"] == "lora") & (df["position"] == "last"))]
    c = curve.groupby("fraction")["test_acc"].agg(["mean", "std"]).reset_index()
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.scatter(curve["fraction"] * 100, curve["test_acc"] * 100, color="#9aa5b1", s=14, zorder=2, label="отдельные seed")
    ax.errorbar(c["fraction"] * 100, c["mean"] * 100, yerr=c["std"] * 100, color="#2a6fdb", marker="o",
                capsize=4, lw=2, zorder=3, label="LoRA в последних k% блоков (mean ± std)")
    full = df[df["mode"] == "full"]["test_acc"]
    if len(full):
        ax.axhline(full.mean() * 100, color="#d1495b", ls="--", lw=1.5, label=f"Full FT ({full.mean()*100:.1f}%)")
    first = df[(df["mode"] == "lora") & (df["position"] == "first")]["test_acc"]
    if len(first):
        ax.errorbar([50], [first.mean() * 100], yerr=[first.std() * 100], color="#e09f3e", marker="s",
                    capsize=4, ls="none", ms=7, zorder=4, label="LoRA в первых 50%")
    ax.set_xlabel("Доля адаптированных блоков, %")
    ax.set_ylabel("Test accuracy, %")
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "fraction_vs_accuracy.png"), dpi=200)

    # --- рисунок 2: trainable params -> качество
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for v, row in g.iterrows():
        col = "#d1495b" if v.startswith("Full") else ("#e09f3e" if "first" in v else "#2a6fdb")
        ax.errorbar(row["params"], row["test_mean"] * 100, yerr=row["test_std"] * 100, marker="o", color=col, capsize=3)
        ax.annotate(v, (row["params"], row["test_mean"] * 100), textcoords="offset points", xytext=(5, -10), fontsize=7)
    ax.set_xscale("log")
    ax.set_xlabel("Число обучаемых параметров (log)")
    ax.set_ylabel("Test accuracy, %")
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "params_vs_accuracy.png"), dpi=200)

    # --- рисунок 3: кривые dev accuracy по эпохам (seed 0)
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for v in g.index:
        r = df[(df["variant"] == v) & (df["seed"] == df["seed"].min())]
        if not len(r):
            continue
        h = json.loads(r.iloc[0]["history"])
        ax.plot([x["epoch"] for x in h], [x["dev_acc"] * 100 for x in h], marker="o", label=v)
    ax.set_xlabel("Эпоха")
    ax.set_ylabel("Dev accuracy, %")
    ax.set_xticks([1, 2, 3])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "dev_curves.png"), dpi=200)


if __name__ == "__main__":
    main()


## Шаг 3. Данные: SST-2
Бинарная тональность отзывов о фильмах. Оригинальное разбиение: train 6920 / dev 872 / test 1821 (у test есть метки). Смотрим размеры, баланс классов и длины в токенах: отсюда `max_len=64`.

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from lora_depth import load_sst2

MODEL = "google/bert_uncased_L-8_H-512_A-8"   # BERT-medium: 8 блоков, hidden 512, ~42M параметров
data = load_sst2("data/sst2")
tok = AutoTokenizer.from_pretrained(MODEL)
for split, (texts, labels) in data.items():
    lens = [len(x) for x in tok(texts)["input_ids"]]
    print(f"{split:5s} n={len(texts):5d}  доля позитивных={np.mean(labels):.3f}  "
          f"длина: медиана={np.median(lens):.0f}, p95={np.percentile(lens,95):.0f}, max={max(lens)}")
print("\nпример:", data["train"][0][0][:100], "-> метка", data["train"][1][0])

## Шаг 4. Модель и конфиг LoRA
- `layers_to_transform` — в какие блоки ставить LoRA (это и есть изучаемый фактор);
- `target_modules = query, key, value, dense` — все линейные слои блока (attention + FFN);
- `modules_to_save = classifier, pooler` — голова учится во всех вариантах одинаково;
- всё остальное заморожено.

Проверяем, что LoRA реально попала только в нужные блоки, и считаем обучаемые параметры.

In [ ]:
import argparse
from lora_depth import build_model, count_params

for frac in [0.25, 0.5, 0.75, 1.0]:
    args = argparse.Namespace(model=MODEL, mode="lora", fraction=frac, position="last", rank=8, alpha=16)
    model, n_layers, layers = build_model(args)
    lora_blocks = sorted({int(n.split("layer.")[1].split(".")[0]) for n, _ in model.named_parameters() if "lora_A" in n})
    tr, tot = count_params(model)
    print(f"доля {frac:4.0%}: layers_to_transform={layers}, LoRA найдена в блоках {lora_blocks}, "
          f"trainable={tr:,} ({100*tr/tot:.2f}%)")

## Шаг 5. Пробный запуск
LoRA в последних 50% блоков, seed 0. Видно loss и dev accuracy по эпохам, в конце строка метрик. Пишется в отдельный `check.csv`, в итоговые результаты не попадает.

In [ ]:
!python lora_depth.py --mode lora --fraction 0.5 --seed 0 --out results/check.csv

## Шаг 6. Вся сетка эксперимента
head only (0%), LoRA last 25/50/75/100%, full fine-tuning, LoRA first 50% (контроль позиции) × seed 0, 1, 2.
На T4 ≈ 15–25 минут. Лог сохраняется в `results/log.txt`.

In [ ]:
!rm -f results/results.csv
!bash run_all.sh 2>&1 | tee results/log.txt | grep -E --line-buffered '^\[ep|test_acc'

## Шаг 7. Метрики по каждому запуску

In [ ]:
import pandas as pd
df = pd.read_csv("results/results.csv", keep_default_na=False)
df[["mode","position","fraction","seed","trainable_params","dev_acc","test_acc","best_epoch","train_time_s"]]

## Шаг 8. Агрегация и графики
Доля блоков → accuracy, trainable params → accuracy, dev accuracy по эпохам. Плюс `results_table.tex` и `numbers.tex` для отчёта.

In [ ]:
!python analyze.py --csv results/results.csv --out figures
from IPython.display import Image, display
for f in ["fraction_vs_accuracy.png", "params_vs_accuracy.png", "dev_curves.png"]:
    display(Image(f"figures/{f}", width=600))

## Шаг 9. Скачать результаты
В архиве `results/` (CSV + лог) и `figures/` (графики, `results_table.tex`, `numbers.tex`). Папку `figures/` положи рядом с `report.tex` и пересобери отчёт — таблица, графики и числа в тексте обновятся сами.

In [ ]:
!zip -qr lora_depth_results.zip results figures
from google.colab import files
files.download("lora_depth_results.zip")